# Agent Lightning：从轨迹回放到步骤级信用分配

**面试问题：只有最终成功奖励时，怎样判断 Agent 的哪一步值得强化？**

## 回答主线

1. Agent 训练首先要把框架执行轨迹标准化为状态、动作、观测、成本和 episode 边界。
2. 把最终奖励复制给每一步会奖励无关工具，也无法定位导致失败的关键动作。
3. 一种可解释基线是做反事实回放：删除某一步，重新计算终态分数，其差值就是该步边际贡献。
4. 正贡献说明动作推进目标，零贡献说明冗余，负贡献通常来自成本、风险或错误副作用。
5. 信用必须限定在同一请求边界内，否则多用户轨迹拼接会产生奖励泄漏。
6. 真实 Agent Lightning 可接更复杂的 RL 算法，但可观测轨迹和可回放奖励函数仍是根基。

## 真实案例

一个退款 Agent 执行核验身份、取消订单、退款、通知用户等六步，其中还误调用了天气工具。我们从事件轨迹重放业务状态，比较“最终奖励复制到所有步骤”的基线与逐步删除反事实；随后构造两个请求被错误拼接的边界泄漏。教学实验使用可读的小数据解释机制，结果不能外推为线上收益。

### 输入预览：六步退款轨迹

In [1]:
from copy import deepcopy  # 导入深拷贝以隔离反事实回放状态。

trajectory = [  # 构造包含必要动作和无关动作的真实 Agent 轨迹。
    {"step": 1, "action": "get_order", "args": {"order_id": "A7842"}, "cost": 0.02},  # 查询订单是后续决策的前提。
    {"step": 2, "action": "verify_identity", "args": {"last4": "4021"}, "cost": 0.03},  # 核验身份以满足退款政策。
    {"step": 3, "action": "lookup_weather", "args": {"city": "上海"}, "cost": 0.08},  # 插入对退款无帮助的错误工具调用。
    {"step": 4, "action": "cancel_order", "args": {"order_id": "A7842"}, "cost": 0.05},  # 取消尚未发货的订单。
    {"step": 5, "action": "refund_payment", "args": {"order_id": "A7842", "amount": 899}, "cost": 0.07},  # 原路发起退款。
    {"step": 6, "action": "send_notification", "args": {"channel": "sms"}, "cost": 0.02},  # 告知用户处理结果。
]  # 完成六步事件序列。
initial_state = {"order_loaded": False, "verified": False, "cancelled": False, "refunded": False, "notified": False, "violations": []}  # 定义可回放业务初态。
print("step  action               cost  args")  # 输出轨迹表头。
for event in trajectory:  # 逐步展示 Agent 实际采取的动作。
    print(f"{event['step']:>2}    {event['action']:<20} {event['cost']:.2f}  {event['args']}")  # 展示工具、参数和成本。
print("目标：verified + cancelled + refunded + notified，且没有策略违规。")  # 明确奖励函数所对应的业务目标。

step  action               cost  args
 1    get_order            0.02  {'order_id': 'A7842'}
 2    verify_identity      0.03  {'last4': '4021'}
 3    lookup_weather       0.08  {'city': '上海'}
 4    cancel_order         0.05  {'order_id': 'A7842'}
 5    refund_payment       0.07  {'order_id': 'A7842', 'amount': 899}
 6    send_notification    0.02  {'channel': 'sms'}
目标：verified + cancelled + refunded + notified，且没有策略违规。


## Baseline 基线：把最终奖励复制给所有步骤

In [2]:
def replay(events):  # 根据动作序列重放退款业务状态。
    state = deepcopy(initial_state)  # 为当前 episode 创建隔离状态。
    total_cost = 0.0  # 累加工具延迟或费用代理。
    for event in events:  # 按发生顺序应用动作。
        action = event["action"]  # 读取当前工具动作。
        total_cost += event["cost"]  # 无论动作是否有用都计入成本。
        if action == "get_order":  # 处理订单查询。
            state["order_loaded"] = True  # 标记订单已载入。
        elif action == "verify_identity" and state["order_loaded"]:  # 只有先查订单才能核验身份。
            state["verified"] = True  # 标记用户已核验。
        elif action == "cancel_order" and state["verified"]:  # 只有核验后才能取消。
            state["cancelled"] = True  # 标记订单已取消。
        elif action == "refund_payment" and state["cancelled"]:  # 只有取消后才能退款。
            state["refunded"] = True  # 标记退款成功。
        elif action == "send_notification" and state["refunded"]:  # 只有退款后通知才算有效。
            state["notified"] = True  # 标记用户已收到通知。
    completed = sum(state[key] for key in ("verified", "cancelled", "refunded", "notified")) / 4.0  # 计算四个目标状态的完成比例。
    score = completed - total_cost  # 用任务完成度减去工具成本形成可解释奖励。
    return state, round(score, 4)  # 返回终态和总奖励。

final_state, final_reward = replay(trajectory)  # 重放完整六步轨迹。
uniform_credit = [final_reward for _ in trajectory]  # 朴素地把最终奖励复制给每个动作。
print("终态：", final_state)  # 展示任务确实成功。
print(f"最终奖励={final_reward:.3f}，复制后的逐步信用={uniform_credit}")  # 展示天气工具也被错误奖励的问题。

终态： {'order_loaded': True, 'verified': True, 'cancelled': True, 'refunded': True, 'notified': True, 'violations': []}
最终奖励=0.730，复制后的逐步信用=[0.73, 0.73, 0.73, 0.73, 0.73, 0.73]


### 核心实现：删除一步的反事实边际贡献

In [3]:
def counterfactual_credits(events):  # 通过逐步删除并回放计算边际信用。
    _, full_score = replay(events)  # 先计算完整轨迹奖励作为参照。
    rows = []  # 收集每一步反事实结果。
    for index, event in enumerate(events):  # 逐动作构造删除反事实。
        without_event = events[:index] + events[index + 1:]  # 删除当前动作并保持其他顺序。
        counterfactual_state, counterfactual_score = replay(without_event)  # 重放删除后的 episode。
        credit = round(full_score - counterfactual_score, 4)  # 用总奖励下降量定义当前动作贡献。
        rows.append({"step": event["step"], "action": event["action"], "without_score": counterfactual_score, "credit": credit, "without_state": counterfactual_state})  # 保存可审计信用记录。
    return rows  # 返回所有动作的信用账本。

credit_rows = counterfactual_credits(trajectory)  # 对退款轨迹执行六次反事实回放。
print("step  action               删除后奖励  边际信用  删除后的关键状态")  # 输出信用分配表头。
for row in credit_rows:  # 逐动作展示反事实贡献。
    state_bits = {key: row["without_state"][key] for key in ("verified", "cancelled", "refunded", "notified")}  # 提取便于阅读的目标状态。
    print(f"{row['step']:>2}    {row['action']:<20} {row['without_score']:>8.3f} {row['credit']:>9.3f}  {state_bits}")  # 展示必要链式动作和无关动作的差异。

step  action               删除后奖励  边际信用  删除后的关键状态
 1    get_order              -0.250     0.980  {'verified': False, 'cancelled': False, 'refunded': False, 'notified': False}
 2    verify_identity        -0.240     0.970  {'verified': False, 'cancelled': False, 'refunded': False, 'notified': False}
 3    lookup_weather          0.810    -0.080  {'verified': True, 'cancelled': True, 'refunded': True, 'notified': True}
 4    cancel_order            0.030     0.700  {'verified': True, 'cancelled': False, 'refunded': False, 'notified': False}
 5    refund_payment          0.300     0.430  {'verified': True, 'cancelled': True, 'refunded': False, 'notified': False}
 6    send_notification       0.500     0.230  {'verified': True, 'cancelled': True, 'refunded': True, 'notified': False}


## 结果解读：有用动作、冗余动作与负信用

In [4]:
credits_by_action = {row["action"]: row["credit"] for row in credit_rows}  # 建立动作到边际信用的索引。
positive_actions = [action for action, credit in credits_by_action.items() if credit > 0.0]  # 收集推进目标的动作。
negative_actions = [action for action, credit in credits_by_action.items() if credit < 0.0]  # 收集因成本拖累奖励的动作。
print("统一复制信用：")  # 输出错误基线概览。
for event, credit in zip(trajectory, uniform_credit):  # 逐动作展示相同最终奖励。
    print(f"{event['action']:<20} credit={credit:.3f}")  # 说明基线无法区分天气调用。
print("反事实正信用动作：", positive_actions)  # 展示任务链中的必要动作。
print("反事实负信用动作：", negative_actions)  # 展示删除后反而更好的无关动作。
print("解读：天气查询不改变终态却增加 0.08 成本，因此信用为负；身份核验被删除后整条取消—退款链都无法完成，贡献最大。")  # 解释信用的因果含义。

统一复制信用：
get_order            credit=0.730
verify_identity      credit=0.730
lookup_weather       credit=0.730
cancel_order         credit=0.730
refund_payment       credit=0.730
send_notification    credit=0.730
反事实正信用动作： ['get_order', 'verify_identity', 'cancel_order', 'refund_payment', 'send_notification']
反事实负信用动作： ['lookup_weather']
解读：天气查询不改变终态却增加 0.08 成本，因此信用为负；身份核验被删除后整条取消—退款链都无法完成，贡献最大。


## 失败案例：跨请求拼接造成奖励泄漏

In [5]:
second_trajectory = [  # 构造第二个独立请求的短轨迹。
    {"step": 1, "action": "get_order", "args": {"order_id": "B9921"}, "cost": 0.02},  # 查询第二个订单。
    {"step": 2, "action": "verify_identity", "args": {"last4": "1188"}, "cost": 0.03},  # 核验第二个用户。
]  # 第二个请求尚未完成取消和退款。
joined_events = trajectory + second_trajectory  # 错误地丢掉 episode_id 并拼接两条轨迹。
_, leaked_score = replay(joined_events)  # 共享状态会让第二请求继承第一请求的成功终态。
_, first_score = replay(trajectory)  # 独立回放第一个已完成请求。
second_state, second_score = replay(second_trajectory)  # 独立回放第二个未完成请求。
boundary_average = round((first_score + second_score) / 2.0, 4)  # 以请求为单位聚合真实奖励。
print(f"错误拼接后单一奖励={leaked_score:.3f}，第二请求终态={second_state}")  # 展示第二请求被第一请求状态污染。
print(f"按 episode 边界独立奖励：first={first_score:.3f} second={second_score:.3f} average={boundary_average:.3f}")  # 展示修正后的评估。
print("修正策略：每个事件携带 trace_id/episode_id，回放器在边界重置环境，并拒绝跨用户状态引用。")  # 给出数据管道门禁。

错误拼接后单一奖励=0.680，第二请求终态={'order_loaded': True, 'verified': True, 'cancelled': False, 'refunded': False, 'notified': False, 'violations': []}
按 episode 边界独立奖励：first=0.730 second=0.200 average=0.465
修正策略：每个事件携带 trace_id/episode_id，回放器在边界重置环境，并拒绝跨用户状态引用。


### 生产边界与训练记录

In [6]:
training_record = {"trace_id": "refund-T101", "reward": final_reward, "credit_method": "leave-one-step-out", "events": len(trajectory), "negative_action": "lookup_weather", "environment_version": "refund-sim-r2"}  # 构造可复现训练样本元数据。
print("训练记录：", training_record)  # 展示算法、环境和轨迹必须一起版本化。
print("生产替换点：真实 Lightning 还需异步 span 采集、模型 logprob、价值函数、批量反事实近似、离策略校正和隐私脱敏。")  # 明确小型确定性回放的边界。

训练记录： {'trace_id': 'refund-T101', 'reward': 0.73, 'credit_method': 'leave-one-step-out', 'events': 6, 'negative_action': 'lookup_weather', 'environment_version': 'refund-sim-r2'}
生产替换点：真实 Lightning 还需异步 span 采集、模型 logprob、价值函数、批量反事实近似、离策略校正和隐私脱敏。


## 回归测试：最后只保护因果信用和 Episode 隔离

In [7]:
assert final_state["refunded"] and final_state["notified"]  # 验证完整轨迹真正完成退款和通知目标。
assert credits_by_action["verify_identity"] > credits_by_action["send_notification"]  # 验证上游关键动作因阻断后续链而贡献更大。
assert credits_by_action["lookup_weather"] < 0.0  # 验证无关且有成本的工具获得负信用。
assert uniform_credit[2] > 0.0  # 验证复制最终奖励的基线确实错误奖励天气动作。
assert second_score < first_score and leaked_score > second_score  # 验证跨请求状态泄漏反例稳定存在。
print("回归测试通过：终态成功、关键动作信用、无关动作惩罚、错误基线和 Episode 隔离均成立。")  # 用少量断言总结信用分配合同。

回归测试通过：终态成功、关键动作信用、无关动作惩罚、错误基线和 Episode 隔离均成立。
